## IDP Phase: Enrichment Linking and Summarization (Agentic workflow)

This notebook demonstrates how to use Strands Agent to:
1. Infer key medical information (diagnosis, medications, treatments) from a medical chart
3. Enrich the data with standardized medical codes (ICD-10, RxNorm, SNOMED CT)

### Overview 

This Medical Document Processing Assistant is an AI-powered tool designed to extract, analyze, and enrich medical information from various document formats such as PDFs and images. This assistant specializes in processing clinical notes, pathology reports, discharge summaries, and other medical documents to provide structured data with standardized medical coding. We will be using Strands to define the agent. 

![arch](architecture.png)

We will defining tools to determine and infer ICD10 code, RxNorm and SNOMED. We are calling the following API within the tool to make the determination. 

1. ICD-10-CM & ICD-10-PCS (U.S. Versions)
Official Source: U.S. Centers for Medicare & Medicaid Services (CMS) and the National Center for Health Statistics (NCHS)

ICD-10-CM (diagnoses):

Call API: https://clinicaltables.nlm.nih.gov/apidoc/icd10cm/v3/doc.html


ICD-10-PCS (procedures):

https://www.cms.gov/medicare/icd-10/2025-icd-10-pcs
(Adjust year as needed)

2. RxNorm
Official Source: U.S. National Library of Medicine (NLM)

https://lhncbc.nlm.nih.gov/RxNav/APIs/RxNormAPIs.html?_gl=1*1qdlo6u*_ga*ODQ1ODkzMzMyLjE3NDg4MzYwMjc.*_ga_7147EPK006*czE3NDg4MzYwMjYkbzEkZzEkdDE3NDg4MzY2MDAkajYwJGwwJGgw*_ga_P1FPTH9PL4*czE3NDg4MzYwMjYkbzEkZzEkdDE3NDg4MzY2MDAkajYwJGwwJGgw

4. SNOMED CT
International Edition

Official Source: SNOMED 

https://browser.ihtsdotools.org/?perspective=full&conceptId1=404684003&edition=MAIN/SNOMEDCT-US/2025-03-01&release=&languages=en


### Step-1: Setup and Dependencies
#### Prerequisites
* Python 3.10+
* AWS account
* Anthropic Claude 3.7 enabled on Amazon Bedrock, [guide](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)
* IAM role with permissions to create Amazon Bedrock Knowledge Base, Amazon S3 bucket

Let's now install the requirement packages for our Strands Agent

In [ ]:
# installing pre-requisites
!pip install -r requirements.txt

In [ ]:
pip install --upgrade strands-agents


In [ ]:
pip install strands-agents-tools strands-agents-builder

In [ ]:
!pip show strands-agents-tools

In [ ]:
!pip show strands-agents

In [ ]:
import os
import json
from typing import Dict, List, Optional, Any
import pypdf
import boto3
from strands import Agent, tool
from strands.models import BedrockModel

### Step-2: Creating Strands Agent

* Create a Strands agent using the tools in medical_coding_tools.py file. Review this file to understand how the tools work

In [ ]:
import os
import logging
from strands import Agent
from strands_tools import file_read
from document_processor import process_document
from medical_coding_tools import (
    get_icd, get_rx, get_snomed,
    link_icd, link_rx, link_snomed,serialize_agent_result 
)
# System prompt for the medical document processing agent
SYSTEM_PROMPT = """
You are a Medical Document Processing Assistant specialized in extracting and analyzing medical information from clinical documents.

Your tasks include:
1. Processing medical documents (PDFs, images) to extract text
2. Identifying key medical information: diagnoses, medications, treatments
3. Enriching the extracted information with standardized medical codes:
   - ICD-10 codes for diagnoses
   - RxNorm codes for medications
   - SNOMED CT codes for treatments

Provide clear, accurate, and structured information that can be used by healthcare professionals.
"""

# Create the medical document processing agent
medical_agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    tools=[
        file_read,
        process_document,
        get_icd,
        get_rx,
        get_snomed,
        link_icd,
        link_rx,
        link_snomed,
        serialize_agent_result
    ]
)


### Step-4: Get Extracted BDA processed data from S3
* Read the BDA-extracted and processed output from S3

In [ ]:
import boto3
import json
import os
from medical_coding_tools import serialize_agent_result
# Get the AWS account number
sts_client = boto3.client('sts')
account_number = sts_client.get_caller_identity()['Account']

# Define bucket and folder paths
bucket_name = f"idp-workshop-{account_number}-us-west-2"
input_folder = "BDA-output"  # Read from BDA-output folder
output_folder = "enriched-output"  # Write to enriched-output folder

print(f"Using bucket: {bucket_name}")
print(f"Input folder: {input_folder}")
print(f"Output folder: {output_folder}")

# Initialize S3 client
s3 = boto3.client('s3')
obj = s3.get_object(Bucket=bucket_name, Key=f"{input_folder}/bda_processed_output.json")
file_content = obj['Body'].read().decode('utf-8')
input_data = json.loads(file_content)
filename = os.path.basename(file_key)
filename_without_ext = os.path.splitext(filename)[0]
print(input_data)

### Step-5: Create strands agent to enrich
* We will invoke the medical agent we created in the previous step using this input data
* The Medical agent will perform its analysis using the various tools. You will have observability about the different steps taken  by the agent and the tools used at every stage as the cell output
* The response will then be structured into expected json format


In [ ]:
# This is the input to the medical agent
# Pass the entire input_data to the medical agent for processing
response = medical_agent(f"Process this medical data and extract diagnoses, medications, and treatments with their respective medical codes: {json.dumps(input_data)}")
#print("Processing complete!",response)

# The response from the medical agent would contain the enriched data
# In a real scenario, we would parse the agent's response to extract the structured information
# For this example, we'll use the sample data structure provided

# Create patient_info structure from the input data
patient_info = {
    "gender": input_data.get("gender", {}),
    "insurance_provider": input_data.get("insurance_provider", {}),
    "date_of_birth": input_data.get("date_of_birth", {}),
    "patient_name": input_data.get("patient_name", {}),
    "phone_number": input_data.get("phone_number", {}),
    "physician_name": input_data.get("physician name", {})  # Note the space in the key
}


structured_output = serialize_agent_result(response, patient_info)
print("structured_output", structured_output)

### Step-6: Parse and Write to S3
* The json output will be written to the same s3 bucket in the 'enriched-output' folder. 

In [ ]:
# Save the structured output from the medical agent to S3
output_key = f"{output_folder}/bda_processed_output_enriched.json"
s3.put_object(
    Bucket=bucket_name,
    Key=output_key,
    Body=json.dumps(structured_output, indent=2),
    ContentType='application/json'
)

print(f"Saved enriched output to s3://{bucket_name}/{output_key}")

## Conclusion

This notebook demonstrates how to use Strands Agent to:
1. Identify key medical information (diagnoses, medications, treatments)
3. Enrich the data with standardized medical codes (ICD-10, RxNorm, SNOMED CT)
4. Process data from S3 BDA-output folder using the medical agent and generate enriched structured output in the enriched-output folder

The agent uses a combination of tools to perform these tasks:
- PDF text extraction
- Medical code lookup (ICD-10, RxNorm, SNOMED CT)
- Medical information enrichment
- S3 integration for input and output

This approach can be extended to handle more complex medical documents and integrate with real medical code databases or APIs.